# From raw/nis/* we will extract data from our .DAT file to generate CSV's and merge into one

### Let's do a quick demo before building the function: ```read_nis```
Using
- raw/nis/2015/NIS-PUF15.SAS
- raw/nis/2015/NISPUF15.DAT

In [1]:
with open('../raw/nis/2015/NIS-PUF15.SAS', "r", encoding="latin1") as f:
        sas_text = f.read()

### We need to parse value and their numerical reps
Example
```
value SEX
. = "MISSING"
1 = "MALE"
2 = "FEMALE"
77 = "DON'T KNOW"
99 = "REFUSED"
```

### Read the .DAT file

In [2]:
import pandas as pd

df = pd.read_fwf(
    "../raw/nis/2015/NISPUF15.DAT",
    colspecs=[(176, 177)],
    names=["SEX"]
)

sex_map = {
    1: "MALE",
    2: "FEMALE",
    77: "DON'T KNOW",
    99: "REFUSED",
}

df["SEX_label"] = df["SEX"].map(sex_map)

df.head()

,SEX,SEX_label
0,1,MALE
1,1,MALE
2,1,MALE
3,2,FEMALE
4,2,FEMALE


### According to ChatGPT, these are the most important columns for vaccine coverage:
| Column         | Level          | What it means                              | Why you need it                                                                     |
| -------------- | -------------- | ------------------------------------------ | ----------------------------------------------------------------------------------- |
| **`SEQNUMC`**  | Child          | Unique child identifier                    | Identifies the individual child/record                                              |
| **`SEQNUMHH`** | Household      | Household identifier                       | Identifies which household the child belongs to; used in survey design              |
| **`STRATUM`**  | Sampling group | Survey sampling stratum                    | Identifies the sampling group; needed for correct SEs/CIs                           |
| **`PROVWT`**   | Child          | Provider-phase survey weight               | Determines how much the child contributes to population estimates                   |
| **`P_UTDMCV`** | Child          | Measles vaccination status                 | `1` = received ≥1 qualifying measles-containing vaccination; `0` = did not          |
| **`P_NUMMMR`** | Child          | Number of measles-containing vaccine doses | Shows how many provider-reported measles-containing vaccinations the child received |
| **`STATE`**    | Geography      | State code                                 | Lets you calculate vaccination coverage by state                                    |
| **`YEAR`**     | Time           | Survey year                                | Lets you calculate and compare coverage over time                                   |

In [3]:
import pandas as pd

colspecs = [
    (6, 11),      # SEQNUMHH
    (12, 32),     # PROVWT_D
    (90, 94),     # STRATUM
    (94, 98),     # YEAR
    (182, 184),   # STATE
    (253, 254),   # P_UTDMCV
]

names = [
    "SEQNUMHH",
    "PROVWT_D",
    "STRATUM",
    "YEAR",
    "STATE",
    "P_UTDMCV",
]

df = pd.read_fwf(
    "../raw/nis/2015/NISPUF15.DAT",
    colspecs=colspecs,
    names=names,
    na_values=["."]
)

df.head()

,SEQNUMHH,PROVWT_D,STRATUM,YEAR,STATE,P_UTDMCV
0,1,77.861750,2017,2015,42,1.0
1,2,NaN,2072,2015,15,NaN
2,3,73.609547,2019,2015,54,1.0
3,4,NaN,2002,2015,25,NaN
4,5,141.333362,2075,2015,16,1.0


In [4]:
len(df)

27592

In [5]:
df.dtypes

SEQNUMHH      int64
PROVWT_D    float64
STRATUM       int64
YEAR          int64
STATE         int64
P_UTDMCV    float64
dtype: object

In [6]:
assert False

AssertionError: 

## Let's start working on this function comprised of four steps:
1. get column row from .SAS
2. parse data and turn to df
3. translate STATE
4. concat df's
4. save total_df as csv

In [1]:
import re
import us
import pandas as pd

def translate_state(sas_text):

    # Get STATE value block
    # Handles both:
    # VALUE STATE
    # VALUE STATEF
    match = re.search(
        r"value\s+STATEF?\b(.*?);",
        sas_text,
        flags=re.IGNORECASE | re.DOTALL
    )

    if match is None:
        raise ValueError("Could not find STATE or STATEF value block")

    state_block = match.group(1)

    # Extract number = "STATE NAME"
    pairs = re.findall(
        r'(\d+)\s*=\s*"([^"]+)"',
        state_block
    )

    # exceptions the us library may not resolve cleanly
    name_fixes = {
        "DISTRICT OF COLUMBIA": "DC",
        "DIST. OF COLUMBIA": "DC",
        "U.S. VIRGIN ISLANDS": "VI",
        "U.S.  VIRGIN ISLANDS": "VI",
    }

    state_map = {}

    for code, state in pairs:

        # remove leading code embedded in label
        # example: " 1  ALABAMA" -> "ALABAMA"
        state = re.sub(
            r"^\s*\d+\s+",
            "",
            state
        ).strip()

        # normalize whitespace
        state = re.sub(r"\s+", " ", state)

        if state in name_fixes:
            state_map[int(code)] = name_fixes[state]
            continue

        result = us.states.lookup(state)

        if result:
            state_map[int(code)] = result.abbr
        else:
            print(f"Not found: {code} = {state}")

    return state_map

### For parsing colmn data we must use different methods
| Years         | SAS INPUT style  |
| ------------- | ---------------- |
| **1995–2004** | `NAME 001 - 006` |
| **2005–2024** | `@1 NAME 6.`     |


In [2]:
import re
import pandas as pd

standard_names = [
    "child_id",
    "household_id",
    "provider_weight",
    "year",
    "state",
    "mcv_coverage",
    "mcv_doses"
]


def parse_dat_to_csv(names, sas_file_path, dat_file_path):

    colspecs = []
    actual_names = []

    with open(sas_file_path, "r", encoding="latin1") as f:
        sas_text = f.read()

    for name in names:

        # ---------------------------------
        # New format: 2005+
        # @13 PROVWT 19.14
        # ---------------------------------
        new_match = re.search(
            rf"@(\d+)\s+({name})\s+\$?(\d+)(?:\.\d*)?",
            sas_text,
            flags=re.IGNORECASE
        )

        if new_match:

            start = int(new_match.group(1)) - 1
            width = int(new_match.group(3))
            end = start + width

            actual_names.append(new_match.group(2))
            colspecs.append((start, end))

            continue

        # ---------------------------------
        # Old format: 1995-2004
        # W0  0022 - 0031 .5
        # ---------------------------------
        old_match = re.search(
            rf"^\s*({name})\s*(\$?)\s*(\d+)\s*-\s*(\d+)",
            sas_text,
            flags=re.IGNORECASE | re.MULTILINE
        )

        if old_match:

            start = int(old_match.group(3)) - 1
            end = int(old_match.group(4))

            actual_names.append(old_match.group(1))
            colspecs.append((start, end))

            continue

        raise ValueError(
            f"Could not find column '{name}' in {sas_file_path}"
        )

    df = pd.read_fwf(
        dat_file_path,
        colspecs=colspecs,
        names=actual_names,
        na_values=["."]
    )

    # translate state
    state_map = translate_state(sas_text)
    df["STATE"] = df["STATE"].map(state_map)

    # standardized column names
    df.columns = standard_names

    return df

### We are going to need to parse these NIS files from 1995 to 2024, but we have to consider the changes in column names
| NIS year      | Provider weight |
| ------------- | --------------- |
| **1995–2001** | `W0`            |
| **2002**      | `WT`            |
| **2003–2004** | `WGT`           |
| **2005–2010** | `PROVWT`        |
| **2011–2017** | `PROVWT_D`      |
| **2018–2024** | `PROVWT_C`      |


In [3]:
from pathlib import Path
import pandas as pd

total_df = pd.DataFrame(columns=standard_names)

folder = Path("../raw/nis")

base_names = [
    "SEQNUMC",
    "SEQNUMHH",
    None,          # weight changes by year
    "YEAR",
    "STATE",
    "P_UTDMCV",
    "P_NUMMMR"
]

for subfolder in sorted(folder.iterdir()):

    if not subfolder.is_dir():
        continue

    year = int(subfolder.name)

    # choose weight column
    if 1995 <= year <= 2001:
        weight_name = "W0"
    elif year == 2002:
        weight_name = "WT"
    elif 2003 <= year <= 2004:
        weight_name = "WGT"
    elif 2005 <= year <= 2010:
        weight_name = "PROVWT"
    elif 2011 <= year <= 2024:
        weight_name = r"PROVWT_\w+"
    else:
        continue

    names = base_names.copy()
    names[2] = weight_name

    dat_file = None
    sas_file = None

    print(f"Working on: {subfolder}")

    for file in subfolder.iterdir():
        if file.is_file():

            if file.suffix.upper() == ".DAT":
                dat_file = str(file)

            elif file.suffix.upper() == ".SAS":
                sas_file = str(file)

    if dat_file and sas_file:
        df = parse_dat_to_csv(
            names,
            sas_file,
            dat_file
        )

        total_df = pd.concat(
            [total_df, df],
            ignore_index=True
        )

Working on: ../raw/nis/1995
Working on: ../raw/nis/1996
Working on: ../raw/nis/1997
Working on: ../raw/nis/1998
Working on: ../raw/nis/1999
Working on: ../raw/nis/2000
Working on: ../raw/nis/2001
Working on: ../raw/nis/2002
Working on: ../raw/nis/2003
Working on: ../raw/nis/2004
Working on: ../raw/nis/2005
Not found: 3 = 
Not found: 7 = 
Not found: 14 = 
Not found: 43 = 
Not found: 52 = 
Working on: ../raw/nis/2006
Not found: 3 = 
Not found: 7 = 
Not found: 14 = 
Not found: 43 = 
Not found: 52 = 
Working on: ../raw/nis/2007
Not found: 3 = 
Not found: 7 = 
Not found: 14 = 
Not found: 43 = 
Not found: 52 = 
Working on: ../raw/nis/2008
Not found: 3 = 
Not found: 7 = 
Not found: 14 = 
Not found: 43 = 
Not found: 52 = 
Working on: ../raw/nis/2009
Not found: 3 = 
Not found: 7 = 
Not found: 14 = 
Not found: 43 = 
Not found: 52 = 
Working on: ../raw/nis/2010
Not found: 3 = 
Not found: 7 = 
Not found: 14 = 
Not found: 43 = 
Not found: 52 = 
Working on: ../raw/nis/2011
Not found: 3 = 
Not found:

In [4]:
total_df.rename(
    columns=lambda x: "PROVWT" if x.startswith("PROVWT_") else x,
    inplace=True
)
total_df.head()

,child_id,household_id,provider_weight,year,state,mcv_coverage,mcv_doses
0,11,1,NaN,1995,CT,NaN,NaN
1,21,2,NaN,1995,CT,NaN,NaN
2,31,3,NaN,1995,CT,NaN,NaN
3,41,4,286.19587,1995,CT,1.0,1.0
4,51,5,247.0033,1995,CT,1.0,1.0


In [5]:
len(total_df)

882534

# Let's further refind total_df to ensure it works well with our measles incidence data ```rada_notebooks/nis_measles_vacc_coverage.csv (Primary coverage)```

In [6]:
total_df.rename(columns={'STATE': 'state', 'YEAR': 'year'}, inplace=True)

In [13]:
len(total_df)

882534

In [7]:
total_df.head()

,child_id,household_id,provider_weight,year,state,mcv_coverage,mcv_doses
0,11,1,NaN,1995,CT,NaN,NaN
1,21,2,NaN,1995,CT,NaN,NaN
2,31,3,NaN,1995,CT,NaN,NaN
3,41,4,286.19587,1995,CT,1.0,1.0
4,51,5,247.0033,1995,CT,1.0,1.0


### Below we group to calculate weigted coverage for each year and state
- P_UTDMCV: child is vaccinated?
- PROVWT: how much the child has contributed to the coverage

### Let's also find out how many children were vaccinated, as per ChatGPT

### 2. Apply Survey Weights (`PROVWT`)

NIS-Child is a sample survey, so each child does not contribute equally to the population estimate. `PROVWT` represents how much each sampled child contributes to the estimated population.

Vaccination coverage is calculated as:

$$
\text{Vaccination Coverage}
=
\frac{
\sum (\text{PROVWT} \times \text{P\_UTDMCV})
}{
\sum \text{PROVWT}
}
$$

Where:

- `P_UTDMCV = 1` → child received ≥1 measles-containing vaccination
- `P_UTDMCV = 0` → child did not
- `PROVWT` → provider-phase survey weight

For example:

| Child | P_UTDMCV | PROVWT |
|---|---:|---:|
| A | 1 | 100 |
| B | 1 | 200 |
| C | 0 | 50 |
| D | 1 | 150 |

$$
\text{Coverage}
=
\frac{
(100 \times 1) + (200 \times 1) + (50 \times 0) + (150 \times 1)
}{
100 + 200 + 50 + 150
}
=
\frac{450}{500}
=
0.90
=
90\%
$$

Therefore, the estimated measles vaccination coverage is **90%**.

In [8]:
from statsmodels.stats.weightstats import DescrStatsW

coverage_df = (
    total_df
    .dropna(subset=["mcv_coverage", "provider_weight"])
    .groupby(["state", "year"])
    .apply(
        lambda x: pd.Series({
            "coverage": DescrStatsW(
                x["mcv_coverage"],
                weights=x["provider_weight"],
                ddof=0
            ).mean,
            "n": len(x),
            "n_vaccinated": (x["mcv_coverage"] == 1).sum()
        }),
        include_groups=False
    )
    .reset_index()
)

coverage_df["coverage_pct"] = round(coverage_df["coverage"] * 100, 2)
coverage_df.drop(columns='coverage', inplace=True)

In [9]:
coverage_df['state'].isna().any() # make sure all rows have a US state

np.False_

In [12]:
len(coverage_df)

1530

In [14]:
coverage_df['year'].min(), coverage_df['year'].max()

(np.int64(1995), np.int64(2024))

In [10]:
coverage_df.head()

,state,year,n,n_vaccinated,coverage_pct
0,AK,1995,194.0,177.0,89.86
1,AK,1996,260.0,225.0,84.55
2,AK,1997,291.0,257.0,87.41
3,AK,1998,34.0,30.0,87.06
4,AK,1999,349.0,321.0,90.67


## Remember that ```coverage_pct``` is WEIGHTED for the first row here is how ChatGPT explains it:

For this row:

| state | year |   n | n_vaccinated | coverage_pct |
| ----- | ---: | --: | -----------: | -----------: |
| AK    | 2015 | 295 |          262 |       89.70% |

Here's what it means:

* **`AK`** — Alaska.
* **`2015`** — NIS-Child 2015 survey data.
* **`n = 295`** — 295 sampled children in Alaska had usable `P_UTDMCV` and `PROVWT` data.
* **`n_vaccinated = 262`** — Of those 295 actual sampled children, **262 had `P_UTDMCV = 1`**, meaning they met the ≥1 measles-containing-vaccine criterion.
* **`coverage_pct = 89.70%`** — After weighting those 295 children using `PROVWT`, the estimated vaccination coverage is **89.70%**.

Notice that the raw sample percentage is:

$$
\frac{262}{295}\times100 = 88.81\%
$$

But your weighted estimate is:

$$
\boxed{89.70\%}
$$

They're different because each child has a different `PROVWT`.

So the best interpretation is:

> **In the 2015 Alaska NIS-Child sample, 262 of 295 children with usable provider data were measles-vaccinated. After applying NIS provider survey weights, estimated measles vaccination coverage was 89.70%.**

That distinction between **262/295 = observed sample** and **89.70% = weighted population estimate** is important for your project.


## Interpreting NIS-Child Vaccination Coverage

The weighted vaccination coverage estimate should **not** be interpreted as:

> 89.70% of the entire Alaska population was vaccinated.

Instead, it should be interpreted as:

> **The estimated measles-containing vaccination coverage among the NIS-Child target population in Alaska in 2015 was 89.70%.**

#### NIS-Child Target Population

NIS-Child is designed to estimate vaccination coverage among **young children aged 19–35 months** in the United States.

Therefore, the weighted coverage estimate represents this population rather than people of all ages.

For example:

- `n = 295` → 295 children with usable vaccination and survey-weight data were included in the Alaska sample.
- `n_vaccinated = 262` → 262 of those sampled children were classified as having received ≥1 qualifying measles-containing vaccination.
- `coverage_pct = 89.70%` → After applying the NIS provider survey weights (`PROVWT`), the estimated measles vaccination coverage among the target population of young children in Alaska was **89.70%**.

The survey weights allow the sampled children to contribute differently to an estimate intended to represent the broader population of children targeted by NIS-Child.

In [11]:
coverage_df.to_csv('../app/data/nis_measles_vacc_coverage.csv', index=False)